# Metadata relationships and nine groups

Relationships between static VM metadata and training-window CPU usage. Inputs are the two raw release files.

In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd
from IPython.display import display, Image

def is_project(path):
    return (path / 'main.py').is_file() and (path / 'src/workload/data.py').is_file()

anchor = Path(os.environ.get('VM_PROJECT_ROOT', str(Path.cwd()))).expanduser().resolve()
ROOT = next((p for p in [anchor, *anchor.parents] if is_project(p)), None)
if ROOT is None:
    matches = [p for p in anchor.iterdir() if p.is_dir() and is_project(p)]
    if len(matches) != 1:
        raise RuntimeError('Open from the checkout/notebooks directory, or set VM_PROJECT_ROOT to the checkout path.')
    ROOT = matches[0]
sys.path.insert(0, str(ROOT / 'src'))
sys.path.insert(0, str(ROOT / 'notebooks'))
from workload.data import load_data, split, validate_predictions

# Set VM_DATA_DIR, or change this variable to the directory with the two raw files.
DATA_DIR = Path(os.environ.get('VM_DATA_DIR', str(ROOT / 'challenge_data'))).expanduser()
data = load_data(DATA_DIR)
n, future_times = split(data)
RESULTS = ROOT / 'src/results'
OUT = ROOT / 'notebooks/outputs'
OUT.mkdir(parents=True, exist_ok=True)
print(f'{n} training timestamps; {len(future_times)} forecast timestamps; raw files: {DATA_DIR}')


## Subscription sizes and category groups

Count VMs per subscription in supplied metadata, including the current VM. Use fixed bins 1–3, 4–20, >20; retain Unknown workloads. These are sample group sizes, not full customer estates.

In [ ]:
counts = data.counts_at(n)
display(pd.DataFrame({'vm_type': data.groups.get_level_values(0), 'subscription_size': data.groups.get_level_values(1), 'vm_count': counts, 'weight': counts / counts.sum()}))

## Why three subscription-size buckets? Fine-bin comparison and three-bin pooling

The fine-grained comparison uses sizes **1, 2, 3, 4–5, 6–10, 11–20, 21–50, 51–100, 100+** for both subscriptions and deployments. The following graph shows the profiles with training CPU mean/median and category counts. Vertical lines show the final cuts at **3 and 20**.

- **1–3** defines a small subscription: fewer than three *other* VMs, including the current VM when counting size.
- **4–20** pools the next three fine bins into a medium-size cohort.
- **>20** pools the larger subscriptions. It distinguishes larger shared groups while keeping the comparison compact.
- Crossing three size buckets with three workload types gives **nine interpretable model groups**, rather than 27 type/fine-size groups or separate models for every subscription.

Mean/median profiles and category composition help explain why size can be informative. They do **not** establish that 3 and 20 are statistically optimal change points. Twenty was an explicit coarse design choice, not a cutoff selected by a leakage-safe boundary-search experiment. Grouping did not establish a fleet accuracy improvement over the direct aggregate model in the direct-versus-grouped comparison. Any future cutoff search must use validation inside training, never the final 24 values. Each group still has only 110 time observations; more VMs do not create more independent time steps.

In [ ]:
from analysis_helpers import subscription_size_analysis
size_profiles = subscription_size_analysis(data, n, OUT)
display(size_profiles['fine_profiles'])
display(size_profiles['three_bins'])
display(Image(filename=str(OUT / 'subscription_size_rationale.png')))

## Category, sharing and resource relationships

In [ ]:
from analysis_helpers import metadata_analysis
connections = metadata_analysis(data, n, OUT)
for name in ['categories', 'sharing', 'resources']:
    print(name)
    display(connections[name])
display(Image(filename=str(OUT / 'metadata_connections.png')))

## Interpret connections carefully

Shared subscription/deployment IDs have higher training CPU than singleton IDs. Category mix and organization dependence can confound the association. Welch p-values are exploratory and unadjusted here, not causal evidence. The 2-core/32-GB bucket has only four VMs, so its large mean is not a reliable sizing rule. Category labels establish neither prices nor service guarantees.

## Source burst gap and lifetime association

The statistic `abs(P95 max − source average) / sqrt(2)` is a descriptive gap. It is not a validated instability score. Source CPU summaries and deletion timestamps can describe periods after training; they are displayed here only and never passed to forecasting.

In [ ]:
display(connections['burst_gap'])
display(connections['lifetime'])

## Unknown VMs and earlier recorded deletion

Unknown VMs have much earlier recorded deletion timestamps. The following analysis compares category summaries and training CPU for records before/at the maximum source boundary. All source creation timestamps in this release are zero, so deletion-day and recorded-lifetime rankings coincide here; this need not hold in another dataset.

A source deletion boundary can be a censored observation, not a confirmed deletion event. These are descriptive comparisons beyond the short reading window, not a model of observed training churn.

In [ ]:
from analysis_helpers import deletion_analysis
deletion = deletion_analysis(data, n, OUT)
display(deletion['categories'])
display(deletion['usage_groups'])
display(deletion['correlations'])
display(Image(filename=str(OUT / 'deletion_usage.png')))

## What the relationship actually shows

Unknown median recorded deletion is **3.64 days**, versus **29.99 days** for Delay-insensitive and Interactive. But Unknown has a training mean CPU of **10.35%**, above Delay-insensitive's **8.26%**: earlier deletion is not equivalent to a low-usage category.

VMs before the maximum boundary have **lower median** training CPU (3.97% versus 5.73%) but **higher mean** (9.90% versus 9.31%). The overall rank correlation is weakly positive (0.0907); within Interactive it is nearly zero (0.0006), and within Unknown it is negative (-0.1767). Category mix, skew and boundary censoring matter. No causal rule or universal low-usage deletion pattern is established.

## Future work: predict deletion risk from information available now

This is a useful hypothesis to investigate. Estimate **probability of deletion in the next 24 hours** or remaining-lifetime distributions, rather than promising an exact deletion time. Compare a category/age-only baseline with additional prefix-only CPU level, idle fraction, variability, recent trend and organization-size features.

First confirm which source records are actual deletion events and which are right-censored at the trace boundary; do not label every maximum timestamp as a real deletion. Obtain longer, less selectively sampled histories with known event flags and varying VM ages. Use chronological landmark validation and account for related VMs in the same subscription/deployment. Features must use only readings preceding the landmark; source lifetime-average/P95 CPU and actual future deletion timestamps are not valid predictors.

This release has about 11 hours of selected complete CPU traces and no observed internal churn, so it is insufficient to establish reliable event-risk performance. Category-specific checks are necessary: the observed usage association even changes direction. A predicted deletion risk should support planning and review, not automatically terminate a VM.